In [ ]:
import os
import cv2
import pickle

IMAGE_DIR = "../dataset/images/train"
LABEL_DIR = "../dataset/labels/train"

OUTPUT_FILE = "ground_truth.pkl"

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.endswith(".jpg")
])

print(f"Total Images : {len(image_files)}")

ground_truth_data = {}

for index, image_name in enumerate(image_files):

    image_path = os.path.join(IMAGE_DIR, image_name)

    image = cv2.imread(image_path)

    if image is None:
        print(f"Could not read {image_name}")
        continue

    img_h, img_w = image.shape[:2]

    label_file = image_name.replace(".jpg", ".txt")
    label_path = os.path.join(LABEL_DIR, label_file)

    objects = []

    if os.path.exists(label_path):

        with open(label_path, "r") as f:

            for line in f:

                values = line.strip().split()

                class_id = int(values[0])

                x_center = float(values[1])
                y_center = float(values[2])
                width = float(values[3])
                height = float(values[4])

                x1 = int((x_center - width / 2) * img_w)
                y1 = int((y_center - height / 2) * img_h)

                x2 = int((x_center + width / 2) * img_w)
                y2 = int((y_center + height / 2) * img_h)

                objects.append({
                    "class": class_id,
                    "bbox": [x1, y1, x2, y2]
                })

    ground_truth_data[image_name] = {
        "width": img_w,
        "height": img_h,
        "objects": objects
    }

    print(
        f"[{index+1}/{len(image_files)}] "
        f"{image_name} --> {len(objects)} objects"
    )

with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(ground_truth_data, f)

print("\nGround truth saved successfully.")
print(f"Output File : {OUTPUT_FILE}")
sample_image = image_files[0]

image = cv2.imread(os.path.join(IMAGE_DIR, sample_image))

class_names = {
    0: "Text",
    1: "Title",
    2: "List",
    3: "Table",
    4: "Figure"
}

for obj in ground_truth_data[sample_image]["objects"]:

    x1, y1, x2, y2 = obj["bbox"]

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    cv2.putText(
        image,
        class_names[obj["class"]],
        (x1, y1 - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 0, 255),
        2
    )

cv2.imshow("Ground Truth", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import os
import cv2
import pickle

IMAGE_DIR = "dataset/images/train"
OUTPUT_FILE = "region_proposals.pkl"

ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.endswith(".jpg")
])

print(f"Total Images : {len(image_files)}")

all_region_proposals = {}
for index, image_name in enumerate(image_files):

    image_path = os.path.join(IMAGE_DIR, image_name)

    image = cv2.imread(image_path)

    if image is None:
        print(f"Could not read {image_name}")
        continue

    ss.setBaseImage(image)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()

    proposals = []

    for (x, y, w, h) in rects:

        proposals.append({
            "x": int(x),
            "y": int(y),
            "width": int(w),
            "height": int(h)
        })

    all_region_proposals[image_name] = proposals

    print(
        f"[{index+1}/{len(image_files)}] "
        f"{image_name} --> {len(proposals)} proposals"
    )
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(all_region_proposals, f)

print("\nRegion proposals saved successfully.")
print(f"Output File : {OUTPUT_FILE}")
first_image = image_files[0]

image = cv2.imread(os.path.join(IMAGE_DIR, first_image))

output = image.copy()

for proposal in all_region_proposals[first_image][:100]:

    x = proposal["x"]
    y = proposal["y"]
    w = proposal["width"]
    h = proposal["height"]

    cv2.rectangle(
        output,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        1
    )

cv2.imshow("Selective Search - First 100 Proposals", output)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
print("hello")